# Домашнее задание: Продвинутый ООП, продолжение

## Проектируем иерархию классов для работы с файлами

* Будем работать в директории `homework_04`
* Будем использовать `*.py` файлы для реализации классов

<br>

**Исходные данные:**

* В проекте мы работаем с **медиа-файлами** (аудио, видео, фото).
* Есть некоторый общий набор данных о файле, необходимый для реализации бизнес-логики (имя, размер, дата создания, владелец...).
* Для каждого типа медиа-файлов есть свой набор метаданных.

<br>

**Задание:**

1. Попробуйте написать классы для работы с медиа-файлами (они будут основой для пользовательского кода остальных команд).
2. Приведите примеры кода, как можно создать, обновить, удалить или провести какое-нибудь действие (конвертация, извлечение фич) над файлом (можно без реализации деталей).
3. Попробуйте дописать классы для работы с файлами, расположенными не на локальном диске (облако, удаленный сервер, s3-like storage).
4. Попробуйте ответить на вопросы: много ли кода придется дописать / переписать при добавлении новых типов файлов и способов их хранения?

<br>

**Суть задания** — именно проектирование классовой иерархии, а не реализация самой логики, поэтому достаточно, например, просто объявить метод `.save(...)` и в комментарии уточнить, что он должен делать, без конкретной реализации.

## Поэтапная реализация

### 1. Попробуйте написать классы для работы с медиа-файлами

In [ ]:
class MediaFile: #инициация родительского класса, параметры = универсальный набор данных, свойственный всем экземплярам
    def __init__(self, name, ext, size, creation_date, owner): 
        self.name = name
        self.ext = ext
        self.size = size
        self.creation_date = creation_date
        self.owner = owner

'''
--- НАСЛЕДОВАНИЕ---
Создаю дочерние классы - AudioFiles, VideoFiles, PictureFiles
Они наследуют параметры родителя: super().__init__(...)
При этом имеют свои собственные параметры: bitrate, ...
'''

class AudioFiles(MediaFile):
    def __init__(self, name, ext, size, creation_date, owner, bitrate):
        super().__init__(name, ext, size, creation_date, owner) #передача общих параметров для всех объектов в родителя для инициализации
        self.bitrate = bitrate #уникальный параметр, свойственный аудио
        
class VideoFiles(MediaFile):
    def __init__(self, name, ext, size, creation_date, owner, resolution):
        super().__init__(name, ext, size, creation_date, owner)
        self.resolution = resolution #уникальный параметр, свойственный видео

class PictureFiles(MediaFile):
    def __init__(self, name, ext, size, creation_date, owner, compression):
        super().__init__(name, ext, size, creation_date, owner)
        self.compression = compression #уникальный параметр, свойственный картинкам

### 2. Приведите примеры кода, как можно создать, обновить, удалить или провести какое-нибудь действие

In [ ]:
class MediaFile:
    def __init__(self, name, ext, size, creation_date, owner):
        self.name = name
        self.ext = ext
        self.size = size
        self.creation_date = creation_date
        self.owner = owner

        self.allowed_extensions = [] #будет переопределен в дочерних классах - у каждого класса свой допустимый формат файлов
        self.is_exists = False #флаг "физического" состояния файла, по умолчанию - файл физически не создан

    def create_file(self):
        # Какой-то код по созданию файла
        self.is_exists = True
        print(f'Файл {self.name} создан пользователем {self.owner}')
    
    def delete_file(self):
        # Проверка состояния файла перед удалением
        if not self.is_exists:
            print(f'Файл {self.name} либо еще не создан, либо уже удален')
            return 
        
        self.is_exists = False
        print(f'Файл "{self.name}" удален')
    
    def update_file(self, **kwargs):
        for property, value in kwargs.items():
            # Проверка наличия в объекте параметров property, переданных в kwargs.keys. Если пройдена - свойство файла меняется
            if not hasattr(self, property):
                raise TypeError(f'Отсутствует свойство "{property}". Файл "{self.name}" не обновлен')
            setattr(self, property, value)
            print(f'Свойство "{property}" изменено на "{value}"')
        
    def convert_file(self, new_ext):
        # Проверка на допустимость формата для конкретного типа фала.
        new_ext = new_ext.lower()
        if new_ext not in self.allowed_extensions:
            raise ValueError(f'Поддерживаемые форматы для конвертации: {self.allowed_extensions}')
        
        # Если пройдена: 1) прверяется идентичность текущему формату
        if self.ext == new_ext:
            print(f'Файл {self.name} уже в формате "{new_ext}')
            return
        
        # Проверка на количество вхождений подстроки ext в строке с наименованием
        if self.name.endswith(self.ext): #чек, что имя файла действительно заканчивается на текущее расширение
            self.name = self.name[:-len(self.ext)] + new_ext #замена текущего расширения: отрезаю от конца строки длину текущего расширения и добавляю новое расширение
        else:
            raise ValueError(f'Имя файла "{self.name}" не заканчивается на расширение "{self.ext}"')
        
        # Если пройдена: 1) обновляется имя файла и файлу присваивается новый формат
        self.name = self.name.replace(self.ext, new_ext)
        self.ext = new_ext
        print(f'Файл сконвертирован в {new_ext}. Новое имя файла: {self.name}')
    
    

class AudioFiles(MediaFile):
    def __init__(self, name, ext, size, creation_date, owner, bitrate):
        super().__init__(name, ext, size, creation_date, owner)
        self.bitrate = bitrate
        self.allowed_extensions = ['.mp3', '.ogg', '.wma'] #форматы, свойственные аудио-файлам
        

class VideoFiles(MediaFile):
    def __init__(self, name, ext, size, creation_date, owner, resolution):
        super().__init__(name, ext, size, creation_date, owner)
        self.resolution = resolution
        self.allowed_extensions = ['.mp4', '.mov', '.avi', 'mkv'] #форматы, свойственные видео-файлам


class PictureFiles(MediaFile):
    def __init__(self, name, ext, size, creation_date, owner, compression):
        super().__init__(name, ext, size, creation_date, owner)
        self.compression = compression
        self.allowed_extensions = ['.png', '.jpg', '.svg'] #форматы, свойственные картинкам


# --- AudioFiles ---

# 1. Объявление экземпляра класса
my_audio = AudioFiles('Queen - Show must go on.mp3', '.mp3', 10, '2026-03-17', 'Admin', 300)
# 2. Создание
my_audio.create_file()

# --- VideoFiles ---

# 1. Объявление экземпляра класса
my_video = VideoFiles('Пираты карибского моря.mp4', '.mp4', 5000, '2026-03-17', 'Admin', '1920x1080')
# 2.  Создание
my_video.create_file()
# 3.  Обновление (меняем владельца и размер)
my_video.update_file(owner = 'User', size=5500)
# 4.  Конвертация
my_video.convert_file('.mov')
# 5. Удаление
my_video.delete_file()

### 3. Попробуйте дописать классы для работы с файлами, расположенными не на локальном диске (облако, удаленный сервер, s3-like storage).


In [ ]:
'''
---РАЗДЕЛЕНИЕ ЛОГИКИ: ---
Отдельно - логика работы с файлами
Отдельно - логика их хранения
'''

In [ ]:
'''
--- АБСТРАКЦИЯ---
Класс-пустышка Storage, задающий стандарт: все дочерние классы - конкретные хранилища - обязаны иметь метод save(...), сохраняющий файл
'''

# Базовый класс выступает в роли интерфейса (см лекцию)
class Storage:
    def save_file(self, name):
        raise NotImplementedError('У класса должен быть метод save_file(self, name)') #если дочерний класс не создаст свой save_file(self, name), программа остановится с ошибкой
       
class S3_storage(Storage):
    #какая-то логика подключения к s3
    def save_file(self, name):
        print(f'Файл {name} сохранен в S3 хранилище')

class Google_storage(Storage):
    #какая-то логика подключения к s3
    def save_file(self, name):
        print(f'Файл {name} сохранен на Google Диск')

'''
---ПОЛИМОРФИЗМ---
К объектам разных классов можно обращаться одним способом, т.к. у них одинаковый метод save(...)
'''

# --- AudioFiles ---

# 1. Объявление экземпляра класса
my_music_storage = Google_storage()
# 2. Сохранение на гуглдиск
my_music_storage.save_file('Queen - Show must go on.mp3')

# --- VideoFiles ---

# 1. Объявление экземпляра класса
my_video_storage = S3_storage()
# 2. Сохранение на с3
my_video_storage.save_file(my_video)


In [ ]:
'''
---ДОПОЛНЕНИЕ ЛОГИКИ---
Связывание файла и хранилки, чтобы файл помнил, где он лежит
'''

In [ ]:
class MediaFile:
    def __init__(self, name, ext, size, creation_date, owner):
        self.name = name
        self.ext = ext
        self.size = size
        self.creation_date = creation_date
        self.owner = owner

        self.allowed_extensions = [] 
        self.is_exists = False 
        self.storage = None #"связь" между логикой самого файла и его хранилкой, хранит ссылку на объект дочернего класса Storages
    
    def get_file_storage(self): #метод для инспекции местоположения файла
        if not self.storage:
            print(f'Файл "{self.name}" еще нигде не сохранен: нужно сохранить файл через хранилище (Storage.save_file())')
            return 
        print(f'Файл "{self.name}" хранится в "{self.storage.__class__.__name__}"') #достаем название метода специальным дандер методом (см. Продвинутый ООП Исключения)

    def delete_file(self): #доработка метода: теперь не просто флаг, а инициатор удаления в связанном объекте-хранилище файла
        if not self.is_exists:
            return f'Файл {self.name} либо еще не создан, либо уже удален'

        if not self.storage:
            print(f'Файл "{self.name}" отсутствует в хранилище, нечего удалять: перед удалением нужно сохранить файл через хранилище (Storage.save_file())')
            return
        
        self.storage.delete_file(self)
        print(f'Файл "{self.name}" удален')
    
    #---не меняла, добавила для отладки---#
    def create_file(self):
        self.is_exists = True
        print(f'Файл {self.name} создан пользователем {self.owner}')

#---не меняла, добавила для отладки---#
class AudioFiles(MediaFile):
    def __init__(self, name, ext, size, creation_date, owner, bitrate):
        super().__init__(name, ext, size, creation_date, owner)
        self.bitrate = bitrate
        self.allowed_extensions = ['.mp3', '.ogg', '.wma']
        
#---не меняла, добавила для отладки---#
class VideoFiles(MediaFile):
    def __init__(self, name, ext, size, creation_date, owner, resolution):
        super().__init__(name, ext, size, creation_date, owner)
        self.resolution = resolution
        self.allowed_extensions = ['.mp4', '.mov', '.avi', 'mkv']

class Storage:
    def save_file(self, media): #name -> media: теперь передаем  не названия файлов, а именно сами файлы-объекты класса MediaFile
        raise NotImplementedError('У класса должен быть метод save_file(self, media)')
    
    def delete_file(self, media):
        raise NotImplementedError('У класса должен быть метод delete_file(self, media)')

class Google_storage(Storage):
    def save_file(self, media):
        if not media.is_exists:
            raise ValueError(f'Нельзя добавить файл "{media.name}" в хранилище, так как он не создан (is_exists=False): для добавления файла в хранилище нужно создать файл через вызов media.create_file()')
        
        media.storage = self #УСТАНОВКА СВЯЗИ файла-объекта и объекта-хранилища: теперь в self.storage MediaFile хранится сам объект-файл
        media.is_exists = True
        print(f'Файл "{media.name}" сохранен на Google Диск')

    def delete_file(self, media):
        media.storage = None #УДАЛЕНИЕ СВЯЗИ файла-объекта и объекта-хранилища
        media.is_exists = False
        print(f'Файл "{media.name}" удален из Google Диска')
       
class S3_storage(Storage):
    def save_file(self, media):
        if not media.is_exists:
            raise ValueError(f'Нельзя добавить файл "{media.name}" в хранилище, так как он не создан (is_exists=False): для добавления файла в хранилище нужно создать файл через вызов media.create_file()')
        
        media.storage = self #УСТАНОВКА СВЯЗИ файла-объекта и объекта-хранилища: теперь в self.storage MediaFile хранится сам объект-файл
        media.is_exists = True
        print(f'Файл "{media.name}" сохранен в S3 хранилище')

    def delete_file(self, media):
        media.storage = None #УДАЛЕНИЕ СВЯЗИ файла-объекта и объекта-хранилища
        media.is_exists = False
        print(f'Файл "{media.name}" удален из S3 хранилища')


# --- AudioFiles ---

# 1. Объявление экземпляра класса
my_audio = AudioFiles('Queen - Show must go on.mp3', '.mp3', 10, '2026-03-17', 'Admin', 300)
# 2. Создание
my_audio.create_file()

# --- VideoFiles ---

# 1. Объявление экземпляра класса
my_video = VideoFiles('Пираты карибского моря.mp4', '.mp4', 5000, '2026-03-17', 'Admin', '1920x1080')
# 2.  Создание
my_video.create_file()

# --- Google_storage ---

# 1. Объявление экземпляра класса
my_music_storage = Google_storage()
# 2. Сохранение САМОГО ОБЪЕКТА my_audio в гугл диск, Google_storage свяжет себя с файлом-объектом my_audio
my_music_storage.save_file(my_audio)
# 3. Получение нахвания хранилища, в котором хранится объект-файл
my_audio.get_file_storage()

# --- S3_storage ---

# 1. Объявление экземпляра класса
my_video_storage = S3_storage()
# 2. Сохранение САМОГО ОБЪЕКТА my_video в с3, S3_storage свяжет себя с файлом-объектом my_video
my_video_storage.save_file(my_video)
# 3. Удаляем файл, меьод сам идет в S3_storage самоудалиться из хранилища
my_video.delete_file()


# Финальный код

In [ ]:
class MediaFile:
    '''
    Базовый класс для работы с медиа-файлами.

    Определяет общие атрибуты (имя, размер, владелец) и базовые операции:
    создание, удаление, обновление метаданных и конвертация.

    Attributes:
        name (str): Имя файла.
        ext (str): Расширение файла (например, '.mp4').
        size (int): Размер файла в условных единицах (МБ/КБ).
        creation_date (str): Дата создания файла.
        owner (str): Владелец/автор файла.
        allowed_extensions (list): Список разрешенных форматов для конвертации.
        is_exists (bool): Статус существования файла.
        storage (Storage): Ссылка на объект хранилища, где размещен файл.
    '''
    def __init__(self, name, ext, size, creation_date, owner):
        '''
        Инициализирует экземпляр файла.

        Args:
            name (str): Имя файла.
            ext (str): Расширение файла (например, '.mp4').
            size (int): Размер файла в условных единицах.
            creation_date (str): Дата создания файла.
            owner (str): Владелец/автор файла.
        '''
        self.name = name
        self.ext = ext
        self.size = size
        self.creation_date = creation_date
        self.owner = owner

        self.allowed_extensions = [] 
        self.is_exists = False 
        self.storage = None

    def create_file(self):
        '''
        Помечает файл как существующий

        Устанавливает флаг `is_exists = True` и выводит сообщение о создании.
        '''
        # Какой-то код по созданию файла
        self.is_exists = True
        print(f'Файл "{self.name}" создан пользователем {self.owner}')
    
    def get_file_storage(self):
        '''
        Выводит информацию о текущем месте хранения файла.

        Если файл не сохранён ни в одном хранилище, выводит соответствующее сообщение.
        '''
        if not self.storage:
            print(f'Файл "{self.name}" еще нигде не сохранен: нужно сохранить файл через хранилище (Storage.save_file())')
            return
        print(f'Файл "{self.name}" хранится в "{self.storage.__class__.__name__}"')

    def delete_file(self): # --- refactor: по всему коду delite_file -> delete_file ---
        '''
        Удаляет файл из хранилища и очищает его метаданные.

        Сначала вызывает метод удаления в связанном хранилище (`storage.delete_file()`),
        которое отвечает за сброс статуса существования (`is_exists`) и очистку ссылки на хранилище (`storage`).

        Выводит сообщение об успешном удалении.
        '''
        if not self.is_exists:
            print(f'Файл "{self.name}" либо еще не создан, либо уже удален')
            return

        if not self.storage:
            print(f'Файл "{self.name}" отсутствует в хранилище, нечего удалять: перед удалением нужно сохранить файл через хранилище (Storage.save_file())')
            return
        
        self.storage.delete_file(self)
        print(f'Файл "{self.name}" удален')
    
    def update_file(self, **kwargs):
        '''
        Обновляет метаданные файла по переданным именованным аргументам.

        Args:
            **kwargs: Ключи (названия атрибутов) и их новые значения.

        Raises:
            TypeError: Если передано имя атрибута, которого нет в классе.
        '''
        for property, value in kwargs.items():
            if not hasattr(self, property):
                raise TypeError(f'Отсутствует свойство "{property}". Файл "{self.name}" не обновлен')
            setattr(self, property, value)
            print(f'Свойство "{property}" изменено на "{value}"')
        
    def convert_file(self, new_ext):
        '''
        Изменяет расширение файла и обновляет его имя.

        Заменяет расширение только если оно находится в конце имени файла.
        Обновляет атрибуты `name` и `ext` при успешной конвертации.

        Args:
            new_ext (str): Целевое расширение (например, '.mov').

        Returns:
            None: При успешной конвертации выводится сообщение через print().
            None: Если расширение совпадает с текущим, выводится сообщение через print() и функция завершается.

        Raises:
            ValueError: Если расширение не входит в список allowed_extensions.

        Notes:
            Метод не возвращает строку — все сообщения выводятся через print().
            Если имя файла содержит подстроку, совпадающую с расширением, не в конце,
            замена может не произойти. Рекомендуется использовать корректные имена файлов.
        '''
        new_ext = new_ext.lower()
        if new_ext not in self.allowed_extensions:
            raise ValueError(f'Поддерживаемые форматы для конвертации: {self.allowed_extensions}')
        
        if self.ext == new_ext:
            print(f'Файл {self.name} уже в формате "{new_ext}')
            return

        if self.name.endswith(self.ext): # --- refactor: изменение логики замены вхождений подстроки ext в строке: расширение заменяется только в конце строки ---
            self.name = self.name[:-len(self.ext)] + new_ext
        else:
            raise ValueError(f'Имя файла "{self.name}" не заканчивается на расширение "{self.ext}"')
        
        self.name = self.name.replace(self.ext, new_ext)
        self.ext = new_ext
        print(f'Файл сконвертирован в формат "{new_ext}". Новое имя файла: {self.name}')

class AudioFiles(MediaFile):
    '''
    Класс для работы с аудио-файлами.

    Наследует базовый функционал MediaFile и добавляет специфические параметры звуковых дорожек.

    Attributes:
        name (str): Имя файла (унаследовано от MediaFile).
        ext (str): Расширение файла (унаследовано от MediaFile).
        size (int): Размер файла в условных единицах (унаследовано от MediaFile).
        creation_date (str): Дата создания файла (унаследовано от MediaFile).
        owner (str): Владелец/автор файла (унаследовано от MediaFile). 
        is_exists (bool): Статус существования файла (унаследовано от MediaFile).
        storage (Storage): Ссылка на объект хранилища (унаследовано от MediaFile).
        bitrate (int): Битрейт аудио в кбит/с (например, 320).
        allowed_extensions (list): Список допустимых расширений для конвертации аудио: ['.mp3', '.ogg', '.wma'].
    '''
    def __init__(self, name, ext, size, creation_date, owner, bitrate):
        '''
        Инициализирует аудио-файл с указанием битрейта.

        Args:
            name (str): Имя файла.
            ext (str): Расширение файла (например, '.mp3').
            size (int): Размер файла в условных единицах.
            creation_date (str): Дата создания файла.
            owner (str): Владелец/автор файла.
            bitrate (int): Битрейт аудио в кбит/с (например, 320).
        '''
        super().__init__(name, ext, size, creation_date, owner)
        self.bitrate = bitrate
        self.allowed_extensions = ['.mp3', '.ogg', '.wma']
        
class VideoFiles(MediaFile):
    '''
    Класс для работы с видео-файлами.

    Наследует базовый функционал MediaFile и добавляет параметры видео-разрешения.

    Attributes:
        name (str): Имя файла (унаследовано от MediaFile).
        ext (str): Расширение файла (унаследовано от MediaFile).
        size (int): Размер файла в условных единицах (унаследовано от MediaFile).
        creation_date (str): Дата создания файла (унаследовано от MediaFile).
        owner (str): Владелец/автор файла (унаследовано от MediaFile).
        is_exists (bool): Статус существования файла (унаследовано от MediaFile).
        storage (Storage): Ссылка на объект хранилища (унаследовано от MediaFile).
        resolution (str): Разрешение видео (например, '1920x1080').
        allowed_extensions (list): Список допустимых расширений для конвертации видео: ['.mp4', '.mov', '.avi', '.mkv'].
    '''
    def __init__(self, name, ext, size, creation_date, owner, resolution):
        '''
        Инициализирует видео-файл с указанием разрешения.

        Args:
            name (str): Имя файла.
            ext (str): Расширение файла (например, '.mp4').
            size (int): Размер файла в условных единицах.
            creation_date (str): Дата создания файла.
            owner (str): Владелец/автор файла.
            resolution (str): Разрешение видео (например, '1920x1080').
        '''
        super().__init__(name, ext, size, creation_date, owner)
        self.resolution = resolution
        self.allowed_extensions = ['.mp4', '.mov', '.avi', '.mkv']

class PictureFiles(MediaFile):
    '''
    Класс для работы с графическими файлами.

    Наследует базовый функционал MediaFile и добавляет параметры сжатия изображения.

    Attributes:
        name (str): Имя файла (унаследовано от MediaFile).
        ext (str): Расширение файла (унаследовано от MediaFile).
        size (int): Размер файла в условных единицах (унаследовано от MediaFile).
        creation_date (str): Дата создания файла (унаследовано от MediaFile).
        owner (str): Владелец/автор файла (унаследовано от MediaFile).
        is_exists (bool): Статус существования файла (унаследовано от MediaFile).
        storage (Storage): Ссылка на объект хранилища (унаследовано от MediaFile).
        compression (int): Уровень сжатия в процентах (например, 90).
        allowed_extensions (list): Список допустимых расширений для конвертации изображений: ['.png', '.jpg', '.svg'].
    '''
    def __init__(self, name, ext, size, creation_date, owner, compression):
        '''
        Инициализирует графический файл с указанием уровня сжатия.

        Args:
            name (str): Имя файла.
            ext (str): Расширение файла (например, '.png').
            size (int): Размер файла в условных единицах.
            creation_date (str): Дата создания файла.
            owner (str): Владелец/автор файла.
            compression (int): Уровень сжатия в процентах (например, 90).
        '''
        super().__init__(name, ext, size, creation_date, owner)
        self.compression = compression
        self.allowed_extensions = ['.png', '.jpg', '.svg']

class Storage:
    '''
    Базовый интерфейс для всех типов хранилищ.

    Определяет обязательные методы для сохранения и удаления объектов медиа‑файлов.
    Сами методы не реализованы и должны быть переопределены в дочерних классах.

    Note:
        Этот класс является абстрактным интерфейсом. При наследовании дочерние классы
        обязаны реализовать методы save_file() и delete_file().
    '''
    def save_file(self, media):
        '''
        Абстрактный метод для сохранения файла в хранилище.

        В дочерних классах должен:
        * физически сохранить файл в соответствующем хранилище;
        * установить связь файла с хранилищем (media.storage = self);
        * обновить статус существования файла (media.is_exists = True).

        Args:
            media (MediaFile): Объект медиа‑файла для сохранения.

        Returns:
            None: Метод не возвращает значение, но изменяет состояние переданного объекта media.

        Raises:
            NotImplementedError: Если метод не переопределён в дочернем классе.
        '''
        raise NotImplementedError('У дочернего класса должен быть реализован метод save_file(self, media). Он должен сохранить файл и установить media.storage = self.')
    
    def delete_file(self, media):
        '''
        Абстрактный метод для физического удаления файла из хранилища.

        В дочерних классах должен:
        * удалить файл из физического хранилища;
        * сбросить связь файла с хранилищем (media.storage = None);
        * обновить статус существования файла (media.is_exists = False).

        Args:
            media (MediaFile): Объект медиа-файла для удаления.

        Returns:
            None: Метод не возвращает значение, но изменяет состояние переданного объекта media.

        Raises:
            NotImplementedError: Если метод не переопределен в дочернем классе.
        '''
        raise NotImplementedError('У дочернего класса должен быть реализован метод delete_file(self, media). Он должен удалить файл и сбросить media.storage = None, media.is_exists = False')

class Google_storage(Storage):
    '''
    Реализация хранилища для работы с Google Drive.

    Наследует абстрактный класс Storage и реализует методы сохранения и удаления файлов.
    '''
    def save_file(self, media):
        '''
        Сохраняет файл на Google Drive и связывает его с текущим хранилищем.

        Проверяет, что файл существует (is_exists=True), затем:
        * устанавливает связь файла с хранилищем (media.storage = self);
        * подтверждает существование файла (media.is_exists = True).

        Args:
            media (MediaFile): Объект медиа‑файла для сохранения.

        Returns:
            None: Метод не возвращает значение, но изменяет состояние переданного объекта media.

        Raises:
            ValueError: Если файл не создан (media.is_exists == False).

        Note:
            Перед сохранением файл должен быть создан через вызов media.create_file().
        '''
        if not media.is_exists: # --- refactor: вызов ValueError при попытке сохранить несуществующий файл ---
            raise ValueError(f'Нельзя добавить файл "{media.name}" в хранилище, так как он не создан (is_exists=False): для добавления файла в хранилище нужно создать файл через вызов media.create_file()')
        
        media.storage = self
        media.is_exists = True
        print(f'Файл "{media.name}" сохранен на Google Диск')

    def delete_file(self, media):
        '''
        Удаляет файл с Google Drive и разрывает связь с объектом.

        Обновляет состояние файла:
        * сбрасывает связь с хранилищем (media.storage = None);
        * отмечает файл как несуществующий (media.is_exists = False).

        Args:
            media (MediaFile): Объект медиа‑файла для удаления.

        Returns:
            None: Метод не возвращает значение, но изменяет состояние переданного объекта media.
        '''
        media.storage = None
        media.is_exists = False # --- refactor: обновление состояния файла в хранилище ---
        print(f'Файл "{media.name}" удален из Google Диска')
       
class S3_storage(Storage):
    '''
    Реализация хранилища для работы с облаком Amazon S3.

    Наследует абстрактный класс Storage и реализует методы сохранения и удаления файлов.
    '''
    def save_file(self, media):
        '''
        Сохраняет файл в S3 и связывает его с текущим хранилищем.

        Проверяет, что файл существует (is_exists=True), затем:
        * устанавливает связь файла с хранилищем (media.storage = self);
        * подтверждает существование файла (media.is_exists = True).

        Args:
            media (MediaFile): Объект медиа‑файла для сохранения.

        Returns:
            None: Метод не возвращает значение, но изменяет состояние переданного объекта media.

        Raises:
            ValueError: Если файл не создан (media.is_exists == False).

        Note:
            Перед сохранением файл должен быть создан через вызов media.create_file().
        '''
        if not media.is_exists: # --- refactor: вызов ValueError при попытке сохранить несуществующий файл ---
            raise ValueError(f'Нельзя добавить файл "{media.name}" в хранилище, так как он не создан (is_exists=False): для добавления файла в хранилище нужно создать файл через вызов media.create_file()')
        
        media.storage = self
        media.is_exists = True
        print(f'Файл "{media.name}" сохранен в S3 хранилище')

    def delete_file(self, media):
        '''
        Удаляет файл из S3 и разрывает связь с объектом.

        Обновляет состояние файла:
        * сбрасывает связь с хранилищем (media.storage = None);
        * отмечает файл как несуществующий (media.is_exists = False).

        Args:
            media (MediaFile): Объект медиа‑файла для удаления.

        Returns:
            None: Метод не возвращает значение, но изменяет состояние переданного объекта media.
        '''
        media.storage = None
        media.is_exists = False # --- refactor: обновление состояния файла в хранилище ---
        print(f'Файл "{media.name}" удален из S3 хранилища')

# --- AudioFiles ---

# 1. Объявление экземпляра класса
my_audio = AudioFiles('Queen - Show must go on.mp3', '.mp3', 10, '2026-03-17', 'Admin', 300)
# 2. Создание
my_audio.create_file()

# --- VideoFiles ---

# 1. Объявление экземпляра класса
my_video = VideoFiles('Пираты карибского моря.mp4', '.mp4', 5000, '2026-03-17', 'Admin', '1920x1080')
# 2.  Создание
my_video.create_file()
# 3.  Обновление (меняем владельца и размер)
my_video.update_file(owner = 'User', size=5500)
# 4.  Конвертация
my_video.convert_file('.mov')
# 5. Удаление
my_video.delete_file()

# --- Google_storage ---

# 1. Объявление экземпляра класса
my_music_storage = Google_storage()
# 2. Сохранение САМОГО ОБЪЕКТА my_audio в гугл диск, Google_storage свяжет себя с файлом-объектом my_audio
my_music_storage.save_file(my_audio)
# 3. Получение нахвания хранилища, в котором хранится объект-файл
my_audio.get_file_storage()

# --- S3_storage ---

# 1. Объявление экземпляра класса
my_video_storage = S3_storage()
# 2. Сохранение САМОГО ОБЪЕКТА my_video в с3, S3_storage свяжет себя с файлом-объектом my_video
my_video_storage.save_file(my_video)
# 3. Удаляем файл 
my_video.delete_file()

Файл "Queen - Show must go on.mp3" создан пользователем Admin
Файл "Пираты карибского моря.mp4" создан пользователем Admin
Свойство "owner" изменено на "User"
Свойство "size" изменено на "5500"
Файл сконвертирован в формат ".mov". Новое имя файла: Пираты карибского моря.mov
Файл "Пираты карибского моря.mov" отсутствует в хранилище, нечего удалять: перед удалением нужно сохранить файл через вызов save_file(self, media)
Файл "Queen - Show must go on.mp3" сохранен на гугл диск
Файл "Queen - Show must go on.mp3" хранится в "Google_storage"
Файл "Пираты карибского моря.mov" сохранен в S3 хранилище
Файл "Пираты карибского моря.mov" удален из S3 хранилища
Файл "Пираты карибского моря.mov" удален
